In [ ]:
import sys
import pathlib

# set pythonpath to the main module directory
module_dir = pathlib.Path("..").parent.resolve().parent
if str(module_dir) not in sys.path:
    sys.path.append(str(module_dir))

In [ ]:
import torch
from src.utils.env import set_seed

set_seed(42)

torch.set_float32_matmul_precision("high")

In [86]:
import json
import torch

meta_folder = "../logs/silent-norm-runs-v1/Qwen2.5-3B-Instruct/oasst2_tulu-v3/model.norm/Qwen2.5-3B-Instruct-KL-0.25-iter1"
# meta_folder = "../logs/silent-norm-runs-v3/Qwen2.5-14B-Instruct/oasst2_tulu-v3/model.norm/Qwen2.5-14B-Instruct-KL-1.0-iter1"
# meta_folder = "../logs/silent-norm-runs-v1/Llama-2-7b-chat-hf/oasst2_tulu-v3/model.layers.21/Llama-2-7b-chat-hf-KL-4.0-iter1"
# meta_folder = "../logs/silent-norm-runs-v1/Phi-3-mini-4k-instruct/oasst2_tulu-v3/model.norm/Phi-3-mini-4k-instruct-KL-1.0-iter1"

metadata_path = pathlib.Path(meta_folder) / "metadata" / "metadata.json"
direction_path = pathlib.Path(meta_folder) / "metadata" / "direction.pt"

metadata = json.load(open(metadata_path, "r"))
direction = torch.load(direction_path, weights_only=True)

In [87]:
from scripts.utils.load_dataset import load_dataset, DatasetName
from src.data import TableLoader

ds_train, ds_val, ds_test = load_dataset(DatasetName.TULU_V3)

dl_train = TableLoader(ds_train, batch_size=8, shuffle=True)
dl_val = TableLoader(ds_val, batch_size=8, shuffle=True)
dl_test = TableLoader(ds_test, batch_size=8, shuffle=True)

INFO 03-23 20:00:09 [scripts.utils.load_dataset:143] Loading dataset from disk at /home/fre.gilad/source/silent_direction/scripts/utils/../../data/tulu-v3


In [88]:
import torch
from scripts.utils.load_model import load_model
from src.model import TargetedModel
from src.aliases import Conv

model, tokenizer = load_model(metadata["model_name"], device_map="cpu")
targeted_model = TargetedModel(model, tokenizer, is_chat=True)
direction = direction.to(targeted_model.device, dtype=targeted_model.dtype)

INFO 03-23 20:00:13 [src.utils.huggingface:29] Using Hugging Face token from environment variable `HF_TOKEN` for login.


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

INFO 03-23 20:00:15 [src.utils.huggingface:29] Using Hugging Face token from environment variable `HF_TOKEN` for login.
INFO 03-23 20:00:17 [scripts.utils.load_model:89] Disabled gradients for all model parameters and set model to eval mode.


In [89]:
model

Qwen2ForCausalLM(
  (model): Qwen2Model(
    (embed_tokens): Embedding(151936, 2048)
    (layers): ModuleList(
      (0-35): 36 x Qwen2DecoderLayer(
        (self_attn): Qwen2Attention(
          (q_proj): Linear(in_features=2048, out_features=2048, bias=True)
          (k_proj): Linear(in_features=2048, out_features=256, bias=True)
          (v_proj): Linear(in_features=2048, out_features=256, bias=True)
          (o_proj): Linear(in_features=2048, out_features=2048, bias=False)
        )
        (mlp): Qwen2MLP(
          (gate_proj): Linear(in_features=2048, out_features=11008, bias=False)
          (up_proj): Linear(in_features=2048, out_features=11008, bias=False)
          (down_proj): Linear(in_features=11008, out_features=2048, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen2RMSNorm((2048,), eps=1e-06)
        (post_attention_layernorm): Qwen2RMSNorm((2048,), eps=1e-06)
      )
    )
    (norm): Qwen2RMSNorm((2048,), eps=1e-06)
    (ro

In [90]:
matrix = model.lm_head.weight
print(matrix.shape)
print(direction.shape)
# we want to check if the direction is in the null-space of the matrix

result = model.lm_head.forward(direction)


# compute result L2 norm
result_norm = torch.norm(result)
print("Result L2 norm:", result_norm.item())
print("Direction L2 norm:", torch.norm(direction).item())

# compute normalized direction and output norm by sqrt(d)
d_in = direction.shape[0]
d_out = result.shape[0]

print("Input dimension:", d_in)
print("Output dimension:", d_out)

normalized_input_norm = torch.norm(direction) / (d_in ** 0.5)
normalized_output_norm = torch.norm(result) / (d_out ** 0.5)

print("Normalized input norm:", normalized_input_norm.item())
print("Normalized output norm:", normalized_output_norm.item())

# compute the ratio between the two

ratio = normalized_output_norm / normalized_input_norm
print("Ratio of normalized output norm to normalized input norm:", ratio.item())


torch.Size([151936, 2048])
torch.Size([2048])
Result L2 norm: 8.4375
Direction L2 norm: 1.0
Input dimension: 2048
Output dimension: 151936
Normalized input norm: 0.0220947265625
Normalized output norm: 0.0216064453125
Ratio of normalized output norm to normalized input norm: 0.9765625


In [91]:
from __future__ import annotations

from dataclasses import dataclass
from typing import Optional, Dict, Any, Tuple

import torch


@dataclass
class NullspaceAnalysis:
    input_dim: int
    output_dim: int
    direction_norm: float
    output_norm: float
    relative_residual_top_sv: float
    rms_gain: float
    top_singular_value: float
    min_singular_value: float
    condition_number: float
    numerical_rank: int
    nullity: int
    null_fraction_of_direction: float
    nonnull_fraction_of_direction: float
    energy_in_small_sv_subspace: Dict[str, float]
    singular_value_quantiles: Dict[str, float]


@dataclass
class DatasetEffectAnalysis:
    num_inputs: int
    input_norm_mean: float
    input_norm_median: float
    output_norm_mean: float
    output_norm_median: float
    output_norm_std: float
    delta_output_norm: float
    delta_over_output_mean: float
    delta_over_output_median: float
    relative_ratio_mean: float
    relative_ratio_median: float
    relative_ratio_p90: float
    relative_ratio_p99: float


def _to_work_dtype(x: torch.Tensor, dtype: torch.dtype = torch.float64) -> torch.Tensor:
    """
    Cast to a stable floating dtype for spectral computations.
    """
    if not x.is_floating_point():
        x = x.float()
    return x.to(dtype=dtype)


def _compute_gram_eigh(matrix: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
    """
    Compute eigen-decomposition of G = A^T A.
    Returns:
        evals: ascending eigenvalues, shape [n]
        evecs: corresponding eigenvectors, shape [n, n]
    """
    A = _to_work_dtype(matrix)
    G = A.T @ A
    G = 0.5 * (G + G.T)  # enforce symmetry numerically
    evals, evecs = torch.linalg.eigh(G)
    evals = torch.clamp(evals, min=0.0)
    return evals, evecs


def _singular_values_from_evals(evals: torch.Tensor) -> torch.Tensor:
    return torch.sqrt(torch.clamp(evals, min=0.0))


def _default_rank_tol(svals: torch.Tensor, m: int, n: int) -> float:
    """
    Numerical rank tolerance similar in spirit to LAPACK:
        tol = max(m, n) * eps * sigma_max
    """
    sigma_max = svals.max()
    eps = torch.finfo(svals.dtype).eps
    return float(max(m, n) * eps * sigma_max)


def _project_onto_subspace(
    vector: torch.Tensor,
    basis: torch.Tensor,
) -> torch.Tensor:
    """
    Project vector onto span(basis columns).
    basis: [n, k] orthonormal columns
    vector: [n]
    """
    if basis.numel() == 0:
        return torch.zeros_like(vector)
    coeffs = basis.T @ vector
    return basis @ coeffs


def analyze_direction_vs_nullspace(
    matrix: torch.Tensor,
    direction: torch.Tensor,
    *,
    rank_tol: Optional[float] = None,
    small_sv_thresholds: Tuple[float, ...] = (1e-1, 1e-2, 1e-3, 1e-4, 1e-5),
    device_for_spectral: Optional[torch.device] = None,
) -> NullspaceAnalysis:
    """
    Analyze whether 'direction' is in or near the nullspace of 'matrix'.

    Args:
        matrix: [m, n]
        direction: [n]
        rank_tol:
            Threshold on singular values for defining the numerical nullspace.
            If None, a default tolerance is used.
        small_sv_thresholds:
            Relative thresholds, interpreted as fractions of sigma_max.
            Example: 1e-3 means singular values <= 1e-3 * sigma_max.
        device_for_spectral:
            Optional device to move matrix/direction to for spectral work.

    Returns:
        NullspaceAnalysis
    """
    if matrix.ndim != 2:
        raise ValueError(f"matrix must be 2D, got shape {tuple(matrix.shape)}")
    if direction.ndim != 1:
        raise ValueError(f"direction must be 1D, got shape {tuple(direction.shape)}")
    if matrix.shape[1] != direction.shape[0]:
        raise ValueError(
            f"shape mismatch: matrix is {tuple(matrix.shape)}, direction is {tuple(direction.shape)}"
        )

    A = matrix
    v = direction

    if device_for_spectral is not None:
        A = A.to(device_for_spectral)
        v = v.to(device_for_spectral)

    m, n = A.shape

    # Stable spectral analysis in float64
    evals, evecs = _compute_gram_eigh(A)
    svals = _singular_values_from_evals(evals)

    sigma_max = float(svals.max().item())
    sigma_min = float(svals.min().item())

    if rank_tol is None:
        rank_tol = _default_rank_tol(svals, m, n)

    numerical_rank = int((svals > rank_tol).sum().item())
    nullity = n - numerical_rank

    # Compute Av and norms
    A_work = _to_work_dtype(A)
    v_work = _to_work_dtype(v)
    Av = A_work @ v_work

    v_norm = float(torch.linalg.norm(v_work).item())
    Av_norm = float(torch.linalg.norm(Av).item())

    if v_norm == 0.0:
        raise ValueError("direction has zero norm")

    relative_residual = Av_norm / (sigma_max * v_norm) if sigma_max > 0 else 0.0
    rms_gain = Av_norm / v_norm

    # Nullspace / non-nullspace decomposition
    null_mask = svals <= rank_tol
    nonnull_mask = ~null_mask

    Q_null = evecs[:, null_mask]
    Q_nonnull = evecs[:, nonnull_mask]

    v_null = _project_onto_subspace(v_work, Q_null)
    v_nonnull = _project_onto_subspace(v_work, Q_nonnull)

    v_null_norm = float(torch.linalg.norm(v_null).item())
    v_nonnull_norm = float(torch.linalg.norm(v_nonnull).item())

    null_fraction = v_null_norm / v_norm
    nonnull_fraction = v_nonnull_norm / v_norm

    # Energy of v inside small-singular-value subspaces
    # We compute fraction of ||v||^2 inside {sigma_i <= rel_thresh * sigma_max}.
    coeffs = evecs.T @ v_work
    coeff_energy = coeffs.square()
    total_energy = float(coeff_energy.sum().item())

    energy_in_small_sv_subspace = {}
    for rel_t in small_sv_thresholds:
        abs_t = rel_t * sigma_max
        mask = svals <= abs_t
        frac = float(coeff_energy[mask].sum().item() / total_energy) if total_energy > 0 else 0.0
        energy_in_small_sv_subspace[f"sv_le_{rel_t:.0e}_times_sigma_max"] = frac

    # Singular value summary
    q_levels = [0.0, 0.01, 0.05, 0.1, 0.5, 0.9, 0.95, 0.99, 1.0]
    q_vals = torch.quantile(svals, torch.tensor(q_levels, dtype=svals.dtype, device=svals.device))
    sv_quantiles = {f"q{int(100*q):02d}": float(v.item()) for q, v in zip(q_levels, q_vals)}

    condition_number = sigma_max / sigma_min if sigma_min > 0 else float("inf")

    return NullspaceAnalysis(
        input_dim=n,
        output_dim=m,
        direction_norm=v_norm,
        output_norm=Av_norm,
        relative_residual_top_sv=relative_residual,
        rms_gain=rms_gain,
        top_singular_value=sigma_max,
        min_singular_value=sigma_min,
        condition_number=condition_number,
        numerical_rank=numerical_rank,
        nullity=nullity,
        null_fraction_of_direction=null_fraction,
        nonnull_fraction_of_direction=nonnull_fraction,
        energy_in_small_sv_subspace=energy_in_small_sv_subspace,
        singular_value_quantiles=sv_quantiles,
    )


@torch.no_grad()
def analyze_direction_relative_to_inputs(
    matrix: torch.Tensor,
    direction: torch.Tensor,
    inputs: torch.Tensor,
    *,
    chunk_size: int = 1024,
    alpha: float = 1.0,
) -> DatasetEffectAnalysis:
    """
    Analyze how large the effect of moving along 'direction' is relative to
    a dataset of natural inputs.

    We compare:
        delta = ||A(alpha * direction)||
    against natural output norms ||A x_i||.

    Args:
        matrix: [m, n]
        direction: [n]
        inputs: [N, n]
        chunk_size: process inputs in chunks
        alpha: perturbation size

    Returns:
        DatasetEffectAnalysis
    """
    if matrix.ndim != 2:
        raise ValueError(f"matrix must be 2D, got shape {tuple(matrix.shape)}")
    if direction.ndim != 1:
        raise ValueError(f"direction must be 1D, got shape {tuple(direction.shape)}")
    if inputs.ndim != 2:
        raise ValueError(f"inputs must be 2D, got shape {tuple(inputs.shape)}")
    if matrix.shape[1] != direction.shape[0]:
        raise ValueError(
            f"shape mismatch: matrix {tuple(matrix.shape)} and direction {tuple(direction.shape)}"
        )
    if inputs.shape[1] != direction.shape[0]:
        raise ValueError(
            f"shape mismatch: inputs {tuple(inputs.shape)} and direction {tuple(direction.shape)}"
        )

    A = matrix
    v = direction
    X = inputs

    work_dtype = torch.float32 if A.dtype in (torch.float16, torch.bfloat16, torch.float32) else torch.float64
    A_work = A.to(dtype=work_dtype)
    v_work = v.to(dtype=work_dtype)
    X_work = X.to(dtype=work_dtype)

    delta_vec = A_work @ (alpha * v_work)
    delta_norm = float(torch.linalg.norm(delta_vec).item())

    output_norms = []
    input_norms = []

    N = X_work.shape[0]
    for start in range(0, N, chunk_size):
        end = min(start + chunk_size, N)
        xb = X_work[start:end]                          # [b, n]
        yb = xb @ A_work.T                              # [b, m]
        output_norms.append(torch.linalg.norm(yb, dim=1).cpu())
        input_norms.append(torch.linalg.norm(xb, dim=1).cpu())

    output_norms = torch.cat(output_norms)
    input_norms = torch.cat(input_norms)

    eps = 1e-12
    relative_ratios = delta_norm / torch.clamp(output_norms, min=eps)

    return DatasetEffectAnalysis(
        num_inputs=int(N),
        input_norm_mean=float(input_norms.mean().item()),
        input_norm_median=float(input_norms.median().item()),
        output_norm_mean=float(output_norms.mean().item()),
        output_norm_median=float(output_norms.median().item()),
        output_norm_std=float(output_norms.std(unbiased=False).item()),
        delta_output_norm=delta_norm,
        delta_over_output_mean=float(delta_norm / max(output_norms.mean().item(), eps)),
        delta_over_output_median=float(delta_norm / max(output_norms.median().item(), eps)),
        relative_ratio_mean=float(relative_ratios.mean().item()),
        relative_ratio_median=float(relative_ratios.median().item()),
        relative_ratio_p90=float(torch.quantile(relative_ratios, 0.90).item()),
        relative_ratio_p99=float(torch.quantile(relative_ratios, 0.99).item()),
    )


def pretty_print_nullspace_analysis(res: NullspaceAnalysis) -> None:
    print("=" * 80)
    print("NULLSPACE ANALYSIS")
    print("=" * 80)
    print(f"Input dim n                          : {res.input_dim}")
    print(f"Output dim m                         : {res.output_dim}")
    print(f"||v||                                : {res.direction_norm:.6f}")
    print(f"||A v||                              : {res.output_norm:.6f}")
    print(f"Relative residual ||Av||/(σ_max||v||): {res.relative_residual_top_sv:.6f}")
    print(f"RMS gain ||Av||/||v||                : {res.rms_gain:.6f}")
    print(f"σ_max                                : {res.top_singular_value:.6f}")
    print(f"σ_min                                : {res.min_singular_value:.6f}")
    print(f"Condition number                     : {res.condition_number:.6f}")
    print(f"Numerical rank                       : {res.numerical_rank}")
    print(f"Nullity                              : {res.nullity}")
    print(f"Fraction of v in numerical nullspace : {res.null_fraction_of_direction:.6f}")
    print(f"Fraction of v outside nullspace      : {res.nonnull_fraction_of_direction:.6f}")
    print("\nEnergy of v in small-singular-value subspaces:")
    for k, val in res.energy_in_small_sv_subspace.items():
        print(f"  {k:35s}: {val:.6f}")
    print("\nSingular value quantiles:")
    for k, val in res.singular_value_quantiles.items():
        print(f"  {k:>4s}: {val:.6f}")
    print("=" * 80)


def pretty_print_dataset_effect_analysis(res: DatasetEffectAnalysis) -> None:
    print("=" * 80)
    print("DATASET-RELATIVE EFFECT ANALYSIS")
    print("=" * 80)
    print(f"Num inputs                           : {res.num_inputs}")
    print(f"Mean ||x||                           : {res.input_norm_mean:.6f}")
    print(f"Median ||x||                         : {res.input_norm_median:.6f}")
    print(f"Mean ||A x||                         : {res.output_norm_mean:.6f}")
    print(f"Median ||A x||                       : {res.output_norm_median:.6f}")
    print(f"Std ||A x||                          : {res.output_norm_std:.6f}")
    print(f"||A(alpha v)||                       : {res.delta_output_norm:.6f}")
    print(f"||A(alpha v)|| / mean ||Ax||         : {res.delta_over_output_mean:.6f}")
    print(f"||A(alpha v)|| / median ||Ax||       : {res.delta_over_output_median:.6f}")
    print(f"Mean per-sample ratio                : {res.relative_ratio_mean:.6f}")
    print(f"Median per-sample ratio              : {res.relative_ratio_median:.6f}")
    print(f"P90 per-sample ratio                 : {res.relative_ratio_p90:.6f}")
    print(f"P99 per-sample ratio                 : {res.relative_ratio_p99:.6f}")
    print("=" * 80)


def analyze_direction_full(
    matrix: torch.Tensor,
    direction: torch.Tensor,
    inputs: Optional[torch.Tensor] = None,
    *,
    rank_tol: Optional[float] = None,
    small_sv_thresholds: Tuple[float, ...] = (1e-1, 1e-2, 1e-3, 1e-4, 1e-5),
    spectral_device: Optional[torch.device] = None,
    chunk_size: int = 1024,
    alpha: float = 1.0,
) -> Dict[str, Any]:
    """
    Full analysis wrapper.
    """
    out: Dict[str, Any] = {}

    ns = analyze_direction_vs_nullspace(
        matrix=matrix,
        direction=direction,
        rank_tol=rank_tol,
        small_sv_thresholds=small_sv_thresholds,
        device_for_spectral=spectral_device,
    )
    out["nullspace"] = ns

    if inputs is not None:
        ds = analyze_direction_relative_to_inputs(
            matrix=matrix,
            direction=direction,
            inputs=inputs,
            chunk_size=chunk_size,
            alpha=alpha,
        )
        out["dataset_effect"] = ds

    return out

In [92]:
matrix = model.lm_head.weight          # [151936, 2048]
direction = direction                  # [2048]

results = analyze_direction_full(
    matrix=matrix,
    direction=direction,
    inputs=None,                       # or your natural activations [N, 2048]
    spectral_device=matrix.device,     # or torch.device("cpu")
)

pretty_print_nullspace_analysis(results["nullspace"])

NULLSPACE ANALYSIS
Input dim n                          : 2048
Output dim m                         : 151936
||v||                                : 1.003599
||A v||                              : 8.442041
Relative residual ||Av||/(σ_max||v||): 0.067632
RMS gain ||Av||/||v||                : 8.411766
σ_max                                : 124.374839
σ_min                                : 2.727986
Condition number                     : 45.592187
Numerical rank                       : 2048
Nullity                              : 0
Fraction of v in numerical nullspace : 0.000000
Fraction of v outside nullspace      : 1.000000

Energy of v in small-singular-value subspaces:
  sv_le_1e-01_times_sigma_max        : 0.966620
  sv_le_1e-02_times_sigma_max        : 0.000000
  sv_le_1e-03_times_sigma_max        : 0.000000
  sv_le_1e-04_times_sigma_max        : 0.000000
  sv_le_1e-05_times_sigma_max        : 0.000000

Singular value quantiles:
   q00: 2.727986
   q01: 5.299686
   q05: 6.627731
   q1

In [93]:
output = model.lm_head(direction)

# compute cosine similarity between output and vector of ones

ones_vec = torch.ones_like(output)
cos_sim = torch.nn.functional.cosine_similarity(output, ones_vec, dim=0)
print("Cosine similarity between output and ones vector:", cos_sim.item())

Cosine similarity between output and ones vector: 0.7734375


In [94]:
# normalize output, print quantiles of values within output afterwards

from math import sqrt


output_normalized = torch.nn.functional.normalize(output, p=2, dim=0) * sqrt(output.shape[0])
print("Quantiles of normalized output values:")
for q in [0.0, 0.25, 0.5, 0.75, 1.0]:
    print(f"  {q}: {torch.quantile(output_normalized.float(), torch.tensor(q).float()).item()}")

Quantiles of normalized output values:
  0.0: -2.5625
  0.25: 0.38671875
  0.5: 0.76171875
  0.75: 1.1796875
  1.0: 2.671875
